# DUAL_MODALITY — phân tích mask từ checkpoint

Notebook này phân tích phân bố mask, specialization image/text, đặc điểm cạnh, tác động lên propagation, can thiệp ranking và quan hệ với full–masked gate. Các kết quả là bằng chứng mô tả/ablation, không tự động mang ý nghĩa nhân quả.

In [ ]:
import json, os, sys
from pathlib import Path

ORIGINAL_CWD = Path.cwd().resolve()
if (ORIGINAL_CWD / 'configs').is_dir():
    SRC_DIR = ORIGINAL_CWD
elif (ORIGINAL_CWD.parent / 'configs').is_dir():
    SRC_DIR = ORIGINAL_CWD.parent
elif (ORIGINAL_CWD / 'PGL' / 'src' / 'configs').is_dir():
    SRC_DIR = ORIGINAL_CWD / 'PGL' / 'src'
else:
    raise RuntimeError('Hãy mở notebook từ PGL/src hoặc PGL/src/mask_analysis.')
sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from mask_analysis.dual_modality_diagnostics import (
    _fixed_validation_triplets, _load_checkpoint
)
from mask_analysis.dual_modality_mask_analysis import (
    collect_gate_snapshot, collect_mask_snapshot,
    controlled_propagation_effects, edge_group_characteristics,
    evaluate_intervention, leave_one_out_content_similarity,
    load_raw_features, per_user_mask_statistics,
    probability_summary, random_mask_baseline,
)
from mask_analysis.dual_modality_mask_swap import compare_rankings
from utils.dataloader import EvalDataLoader

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

## 0. Cấu hình phân tích
Đặt checkpoint đầy đủ do trainer PGL lưu (`.pth`, có `config` và `model_state_dict`). Các permutation giữ nguyên toàn bộ phân bố trọng số/số cạnh trong lịch sử từng user.

In [ ]:
CHECKPOINT = r'saved/PUT_DUAL_MODALITY_CHECKPOINT_HERE.pth'
SPLIT = 'test'                 # 'valid' hoặc 'test'
TOPK = [20]
FORCE_CPU = False
SEED = 2024
PERMUTATION_REPEATS = 5       # tăng lên 20+ cho báo cáo cuối
RANDOM_BASELINE_REPEATS = 100
COMPUTE_CONTENT_FIT = True    # phần tốn thời gian nhất với image feature lớn
METADATA_FILE = None          # ví dụ r'../data/baby/item_metadata.csv'
METADATA_ITEM_COLUMN = None   # mặc định dùng ITEM_ID_FIELD
CATEGORY_COLUMN = None        # tên cột category trong metadata
OUTPUT_DIR = None             # mặc định: <checkpoint>-mask-analysis/

In [ ]:
checkpoint_path = Path(CHECKPOINT).expanduser()
if not checkpoint_path.is_absolute():
    candidates = [ORIGINAL_CWD / checkpoint_path, SRC_DIR / checkpoint_path]
    checkpoint_path = next((p for p in candidates if p.is_file()), candidates[-1])
checkpoint_path = checkpoint_path.resolve()
if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)

output_dir = (
    Path(OUTPUT_DIR).expanduser().resolve() if OUTPUT_DIR
    else checkpoint_path.with_suffix('').with_name(checkpoint_path.stem + '-mask-analysis')
)
output_dir.mkdir(parents=True, exist_ok=True)

config, model, train_dataset, valid_dataset, test_dataset = _load_checkpoint(
    str(checkpoint_path), FORCE_CPU
)
if model.mask_sharing_mode != 'separate':
    raise ValueError('Phân tích image/text mask yêu cầu mask_sharing_mode=separate.')
config['topk'] = sorted(set(TOPK))
evaluation_dataset = valid_dataset if SPLIT == 'valid' else test_dataset
eval_data = EvalDataLoader(
    config, evaluation_dataset, additional_dataset=train_dataset,
    batch_size=config['eval_batch_size'],
)

def finish_figure(name):
    plt.tight_layout()
    plt.savefig(output_dir / f'{name}.png', dpi=180, bbox_inches='tight')
    plt.show()

print('Checkpoint:', checkpoint_path)
print('Dataset/split:', config['dataset'], SPLIT)
print('Mask:', model.mask_generation_mode, model.mask_graph_mode, model.mask_degree_mode)
print('Output:', output_dir)

## 1. Phân bố xác suất và mức giữ cạnh theo user
Với hard mask, `selected` là cạnh thực sự dùng khi evaluation. Với soft mask, notebook dùng **global top-k probability proxy** chỉ để tạo bốn nhóm và so budget; propagation thực tế vẫn dùng mọi cạnh với trọng số mềm.

In [ ]:
snapshot = collect_mask_snapshot(model)
user_stats = per_user_mask_statistics(
    snapshot, model.n_users, eps=model.mask_specialization_eps
)
probability_table = pd.DataFrame({
    modality: probability_summary(snapshot['probabilities'][modality].numpy())
    for modality in ('image', 'text')
}).T
display(probability_table)
print('Selection interpretation:', snapshot['selection_kind'])
print('User mất toàn bộ image edges:', user_stats['image_zero_edge_user_rate'])
print('User mất toàn bộ text edges:', user_stats['text_zero_edge_user_rate'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, modality, color in zip(axes, ('image', 'text'), ('tab:blue', 'tab:orange')):
    ax.hist(snapshot['probabilities'][modality].numpy(), bins=50, alpha=.8, color=color)
    ax.axvline(0.05, color='black', linestyle='--', linewidth=1)
    ax.axvline(0.95, color='black', linestyle='--', linewidth=1)
    ax.set(title=f'{modality}: p_ui', xlabel='mask probability', ylabel='edge count')
finish_figure('01_mask_probability_distributions')

In [ ]:
user_df = pd.DataFrame({
    'user': np.arange(model.n_users),
    'activity': user_stats['degree'],
    'image_kept': user_stats['image_kept'],
    'text_kept': user_stats['text_kept'],
    'image_keep_rate': user_stats['image_keep_rate'],
    'text_keep_rate': user_stats['text_keep_rate'],
    'jaccard': user_stats['jaccard'],
    'both_empty': user_stats['both_empty'],
    'user_js': user_stats['per_user_js'],
    'rank_gap': user_stats['normalized_rank_gap'],
    'rank_correlation': user_stats['rank_correlation'],
})
for modality in ('image', 'text'):
    for statistic, values in user_stats[f'{modality}_probability_statistics'].items():
        user_df[f'{modality}_probability_{statistic}'] = values
for group, values in user_stats['group_counts'].items():
    user_df[f'{group}_count'] = values
user_df = user_df[user_df.activity > 0].copy()
bins = [0, 2, 5, 10, 20, 50, np.inf]
labels = ['1-2', '3-5', '6-10', '11-20', '21-50', '51+']
user_df['activity_bin'] = pd.cut(user_df.activity, bins=bins, labels=labels)
display(user_df.groupby('activity_bin', observed=True)[
    ['activity', 'image_kept', 'text_kept', 'image_keep_rate', 'text_keep_rate']
].agg(['count', 'mean', 'median']))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
user_df.boxplot(column='image_keep_rate', by='activity_bin', ax=axes[0], showfliers=False)
user_df.boxplot(column='text_keep_rate', by='activity_bin', ax=axes[1], showfliers=False)
zero_rates = [user_stats['image_zero_edge_user_rate'], user_stats['text_zero_edge_user_rate']]
axes[2].bar(['image', 'text'], zero_rates, color=['tab:blue', 'tab:orange'])
axes[2].set(title='User không còn cạnh', ylabel='fraction of train users', ylim=(0, 1))
axes[0].set(title='Image keep rate theo activity', xlabel='train interactions', ylabel='k_u / |N(u)|')
axes[1].set(title='Text keep rate theo activity', xlabel='train interactions', ylabel='k_u / |N(u)|')
fig.suptitle('')
finish_figure('02_user_budget_by_activity')

# Phân bố soft weight theo activity, độc lập với hard/soft propagation mode.
edge_soft_df = pd.DataFrame({
    'activity': user_stats['degree'][snapshot['edge_users'].numpy()],
    'image_probability': snapshot['probabilities']['image'].numpy(),
    'text_probability': snapshot['probabilities']['text'].numpy(),
})
edge_soft_df['activity_bin'] = pd.cut(edge_soft_df.activity, bins=bins, labels=labels)
soft_activity_summary = user_df.groupby('activity_bin', observed=True)[[
    'image_probability_mean', 'image_probability_std',
    'text_probability_mean', 'text_probability_std'
]].agg(['mean', 'median', 'std'])
display(soft_activity_summary)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
edge_soft_df.boxplot(column='image_probability', by='activity_bin', ax=axes[0], showfliers=False)
edge_soft_df.boxplot(column='text_probability', by='activity_bin', ax=axes[1], showfliers=False)
axes[0].set(title='Image soft weights theo user activity', xlabel='train interactions', ylabel='p_ui')
axes[1].set(title='Text soft weights theo user activity', xlabel='train interactions', ylabel='p_ui')
fig.suptitle('')
finish_figure('02b_soft_weight_distribution_by_activity')

## 2. Hai mask khác nhau ở đâu?
Bốn nhóm cạnh và Jaccard được so với random mask có **đúng số cạnh giữ của từng modality cho từng user**. User có union rỗng được đếm riêng và không được gán Jaccard bằng 0.

In [ ]:
random_df = pd.DataFrame(random_mask_baseline(
    snapshot, user_stats, repeats=RANDOM_BASELINE_REPEATS, seed=SEED
))
groups = ['both', 'image_only', 'text_only', 'neither']
observed_rates = pd.Series(user_stats['global_group_rates']).reindex(groups)
random_means = random_df[groups].mean()
random_stds = random_df[groups].std()
comparison_table = pd.DataFrame({
    'observed': observed_rates, 'random_mean': random_means,
    'random_std': random_stds, 'observed_minus_random': observed_rates-random_means,
})
display(comparison_table)
print('Both-empty users (observed):', user_df.both_empty.mean())
print('Both-empty users (random, same per-user budgets):', random_df.both_empty_user_rate.mean())

x = np.arange(len(groups)); width = .36
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(x-width/2, observed_rates, width, label='observed')
axes[0].bar(x+width/2, random_means, width, yerr=random_stds, label='random ± sd')
axes[0].set_xticks(x, groups, rotation=20); axes[0].set_ylabel('edge fraction')
axes[0].set_title('Bốn nhóm cạnh: observed vs budget-matched random'); axes[0].legend()
valid_j = user_df.jaccard.dropna()
axes[1].hist(valid_j, bins=30, alpha=.75, label='observed user Jaccard')
axes[1].axvline(random_df.mean_user_jaccard.mean(), color='red', linestyle='--',
                label='random mean across repeats')
axes[1].set(title='Jaccard theo user (union không rỗng)', xlabel='J_u', ylabel='users')
axes[1].legend()
finish_figure('03_overlap_vs_random')

In [ ]:
group_rate_by_activity = user_df.groupby('activity_bin', observed=True)[
    [f'{g}_count' for g in groups]
].sum()
group_rate_by_activity = group_rate_by_activity.div(group_rate_by_activity.sum(axis=1), axis=0)
ax = group_rate_by_activity.plot(kind='bar', stacked=True, figsize=(10, 4), colormap='tab20c')
ax.set(title='Phân bố nhóm cạnh theo mức activity của user', xlabel='train interactions', ylabel='edge fraction')
ax.legend(groups, bbox_to_anchor=(1.02, 1), loc='upper left')
finish_figure('04_edge_groups_by_user_activity')

image_p = snapshot['probabilities']['image'].numpy()
text_p = snapshot['probabilities']['text'].numpy()
sample_rng = np.random.default_rng(SEED)
sample = sample_rng.choice(len(image_p), min(50000, len(image_p)), replace=False)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(image_p[sample], text_p[sample], s=3, alpha=.15)
axes[0].set(title=f"Soft probabilities, Pearson={user_stats['global_probability_correlation']:.3f}",
            xlabel='image p_ui', ylabel='text p_ui')
axes[1].hist(user_df.loc[user_df.activity >= 2, 'user_js'], bins=40)
axes[1].axvline(model.mask_specialization_margin, color='red', linestyle='--', label='delta')
axes[1].set(title='JS theo user', xlabel='JS(q_image || q_text)', ylabel='users'); axes[1].legend()
axes[2].hist(user_df.rank_gap.dropna(), bins=40)
axes[2].set(title='Khác biệt thứ hạng cạnh', xlabel='mean normalized |rank_v-rank_t|', ylabel='users')
finish_figure('05_soft_mask_difference')

## 3. Đặc điểm của cạnh riêng theo modality
Content fit dùng feature `.npy` gốc và loại chính item đang xét khỏi lịch sử trung bình. Kết quả này mô tả nội dung được chọn; nó chưa chứng minh preference hay tác động nhân quả.

In [ ]:
edge_characteristics = edge_group_characteristics(snapshot, user_stats, model.n_items)
edge_df = pd.DataFrame(edge_characteristics)
edge_df['log_item_popularity'] = np.log1p(edge_df.item_popularity)
edge_df['log_user_activity'] = np.log1p(edge_df.user_activity)

if COMPUTE_CONTENT_FIT:
    raw_image_features, raw_text_features = load_raw_features(config, train_dataset)
    edge_df['image_content_fit'] = leave_one_out_content_similarity(
        edge_characteristics['edge_users'], edge_characteristics['edge_items'], raw_image_features
    )
    edge_df['text_content_fit'] = leave_one_out_content_similarity(
        edge_characteristics['edge_users'], edge_characteristics['edge_items'], raw_text_features
    )

summary_columns = ['item_popularity', 'user_activity']
if COMPUTE_CONTENT_FIT:
    summary_columns += ['image_content_fit', 'text_content_fit']
display(edge_df.groupby('group')[summary_columns].agg(['count', 'mean', 'median', 'std']))

def grouped_boxplot(ax, frame, column, title):
    data = [frame.loc[frame.group == group, column].dropna().values for group in groups]
    ax.boxplot(data, labels=groups, showfliers=False)
    ax.tick_params(axis='x', rotation=20); ax.set_title(title)

plot_columns = [('log_item_popularity', 'log(1 + item popularity)'),
                ('log_user_activity', 'log(1 + user activity)')]
if COMPUTE_CONTENT_FIT:
    plot_columns += [('image_content_fit', 'leave-one-out image fit'),
                     ('text_content_fit', 'leave-one-out text fit')]
fig, axes = plt.subplots(1, len(plot_columns), figsize=(5*len(plot_columns), 4))
axes = np.atleast_1d(axes)
for ax, (column, title) in zip(axes, plot_columns): grouped_boxplot(ax, edge_df, column, title)
finish_figure('06_edge_group_characteristics')

In [ ]:
# So sánh content fit sau khi chia tầng popularity; tránh kết luận chỉ từ popularity khác nhau.
if COMPUTE_CONTENT_FIT:
    edge_df['popularity_stratum'] = pd.qcut(
        edge_df.item_popularity.rank(method='first'), 4,
        labels=['Q1 low', 'Q2', 'Q3', 'Q4 high']
    )
    stratified = edge_df.groupby(
        ['popularity_stratum', 'group'], observed=True
    )[['image_content_fit', 'text_content_fit']].mean().reset_index()
    display(stratified)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for ax, column, title in zip(axes, ['image_content_fit', 'text_content_fit'],
                                 ['Image fit within popularity strata', 'Text fit within popularity strata']):
        pivot = stratified.pivot(index='popularity_stratum', columns='group', values=column)
        pivot.reindex(columns=groups).plot(kind='bar', ax=ax)
        ax.set_title(title); ax.set_ylabel('mean leave-one-out cosine'); ax.tick_params(axis='x', rotation=20)
    finish_figure('07_content_fit_stratified_by_popularity')
    # Joint stratification theo cả popularity và user activity.
    edge_df['activity_stratum'] = pd.qcut(
        edge_df.user_activity.rank(method='first'), 4,
        labels=['A1 low', 'A2', 'A3', 'A4 high']
    )
    joint_strata = edge_df.groupby(
        ['popularity_stratum', 'activity_stratum', 'group'], observed=True
    )[['image_content_fit', 'text_content_fit']].mean().reset_index()
    contrast_rows = []
    for (pop_bin, activity_bin), frame in joint_strata.groupby(
        ['popularity_stratum', 'activity_stratum'], observed=True
    ):
        indexed = frame.set_index('group')
        if {'image_only', 'text_only'}.issubset(indexed.index):
            contrast_rows.append({
                'popularity': str(pop_bin), 'activity': str(activity_bin),
                'image_fit_image_only_minus_text_only': indexed.loc['image_only', 'image_content_fit']-indexed.loc['text_only', 'image_content_fit'],
                'text_fit_image_only_minus_text_only': indexed.loc['image_only', 'text_content_fit']-indexed.loc['text_only', 'text_content_fit'],
            })
    controlled_contrasts = pd.DataFrame(contrast_rows)
    display(controlled_contrasts)
    if not controlled_contrasts.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.boxplot([
            controlled_contrasts.image_fit_image_only_minus_text_only.dropna(),
            controlled_contrasts.text_fit_image_only_minus_text_only.dropna(),
        ], labels=['image content fit', 'text content fit'], showmeans=True)
        ax.axhline(0, color='black', linewidth=1)
        ax.set(title='Image-only minus text-only within popularity × activity strata', ylabel='mean cosine difference')
        finish_figure('07b_content_fit_controlled_popularity_activity')

# Metadata/category tùy chọn. File phải có một dòng hoặc giá trị category cho mỗi item ID.
if METADATA_FILE and CATEGORY_COLUMN:
    metadata = pd.read_csv(METADATA_FILE, sep=None, engine='python')
    item_column = METADATA_ITEM_COLUMN or config['ITEM_ID_FIELD']
    metadata = metadata[[item_column, CATEGORY_COLUMN]].drop_duplicates(item_column)
    metadata = metadata.rename(columns={item_column: 'edge_items'})
    metadata_edges = edge_df.merge(metadata, on='edge_items', how='left')
    top_categories = metadata_edges[CATEGORY_COLUMN].value_counts().head(15).index
    category_table = pd.crosstab(
        metadata_edges.loc[metadata_edges[CATEGORY_COLUMN].isin(top_categories), 'group'],
        metadata_edges.loc[metadata_edges[CATEGORY_COLUMN].isin(top_categories), CATEGORY_COLUMN],
        normalize='index',
    ).reindex(groups)
    display(category_table)
    category_table.T.plot(kind='bar', figsize=(14, 5))
    plt.ylabel('fraction within mask group'); plt.title('Category composition by edge group')
    finish_figure('08_category_by_edge_group')
else:
    print('Bỏ qua category: đặt METADATA_FILE và CATEGORY_COLUMN nếu dataset có metadata.')

## 4. Mask có thay đổi thông tin truyền trên graph?
Đối chứng dưới đây dùng cùng initial embedding của masked branch cho cả full adjacency và masked adjacency. Vì vậy khác biệt output không còn bị trộn với khác biệt giữa hai user embedding table ban đầu. Với `soft + degree=full`, notebook thêm constant-mask có cùng mean probability.

In [ ]:
propagation_effects = controlled_propagation_effects(model)
propagation_rows = []
for modality, controls in propagation_effects.items():
    for control, values in controls.items():
        if not isinstance(values, dict): continue
        propagation_rows.append({
            'modality': modality, 'control': control,
            'mean_cosine_distance': np.mean(values['cosine_distance']),
            'median_cosine_distance': np.median(values['cosine_distance']),
            'mean_norm_ratio': np.mean(values['norm_ratio']),
            'median_norm_ratio': np.median(values['norm_ratio']),
        })
display(pd.DataFrame(propagation_rows))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for column, modality in enumerate(('image', 'text')):
    for control, values in propagation_effects[modality].items():
        if not isinstance(values, dict): continue
        axes[0, column].hist(values['cosine_distance'], bins=40, alpha=.55, label=control)
        axes[1, column].hist(values['norm_ratio'], bins=40, alpha=.55, label=control)
    axes[0, column].set(title=f'{modality}: direction change', xlabel='1 - cosine', ylabel='users')
    axes[1, column].set(title=f'{modality}: magnitude change', xlabel='||H_alt|| / ||H_full||', ylabel='users')
    axes[0, column].legend(); axes[1, column].legend()
finish_figure('09_controlled_propagation_effects')

## 5. Can thiệp mask và đánh giá ranking
Mỗi permutation chỉ xáo trộn mask trong lịch sử của cùng user, nên giữ nguyên số cạnh hard-mask và phân bố soft weights của user đó. `constant_mask` (chỉ soft mode) thay mỗi mask bằng mean learned probability của chính modality; `swapped` đổi image/text mask; `full_adjacency` thay cả hai masked adjacency bằng full adjacency. Masked-only được đo trước fusion/I–I; final dùng toàn bộ mô hình.

In [ ]:
uid_field, iid_field = train_dataset.uid_field, train_dataset.iid_field
history_by_user = {
    int(user): set(int(item) for item in group[iid_field].values)
    for user, group in train_dataset.df.groupby(uid_field)
}
triplets = _fixed_validation_triplets(
    eval_data.get_eval_users().numpy(), eval_data.get_eval_items(),
    history_by_user, model.n_items, SEED
).to(model.device)
print('Fixed evaluation triplets:', triplets.shape[1])

intervention_results = []
base_interventions = ['original']
if model.mask_graph_mode == 'soft': base_interventions.append('constant_mask')
base_interventions += ['swapped', 'full_adjacency']
for name in base_interventions:
    print('Running', name)
    intervention_results.append(evaluate_intervention(
        model, eval_data, config, triplets, intervention=name, seed=SEED
    ))
for repeat in range(PERMUTATION_REPEATS):
    for name in ('permute_image', 'permute_text'):
        run_seed = SEED + repeat
        print('Running', name, run_seed)
        intervention_results.append(evaluate_intervention(
            model, eval_data, config, triplets, intervention=name, seed=run_seed
        ))
model.set_mask_assignment_mode('normal')

In [ ]:
metric_rows, margin_rows = [], []
for result in intervention_results:
    label = result['intervention']
    metric_rows.append({'intervention': label, 'seed': result['seed'], **result['metrics']})
    row = {'intervention': label, 'seed': result['seed']}
    for branch, values in result['triplets']['summary'].items():
        for metric, value in values.items(): row[f'{branch}_{metric}'] = value
    margin_rows.append(row)
metric_df, margin_df = pd.DataFrame(metric_rows), pd.DataFrame(margin_rows)
display(metric_df)
display(margin_df)
if model.mask_graph_mode == 'soft':
    learned_row = metric_df[metric_df.intervention == 'original'].iloc[0]
    constant_row = metric_df[metric_df.intervention == 'constant_mask'].iloc[0]
    learned_vs_constant = pd.DataFrame({
        'learned': learned_row[[c for c in metric_df if '@' in c]],
        'constant': constant_row[[c for c in metric_df if '@' in c]],
    })
    learned_vs_constant['constant_minus_learned'] = (
        learned_vs_constant['constant'] - learned_vs_constant['learned']
    )
    display(learned_vs_constant)
    constant_result = next(r for r in intervention_results if r['intervention'] == 'constant_mask')
    print('Constant probabilities matched to learned global means:',
          constant_result['mask_probability_means'])

metric_columns = [c for c in metric_df if '@' in c]
fig, axes = plt.subplots(1, len(metric_columns), figsize=(5*len(metric_columns), 4))
axes = np.atleast_1d(axes)
families = ['original'] + (['constant_mask'] if model.mask_graph_mode == 'soft' else []) + [
    'permute_image', 'permute_text', 'swapped', 'full_adjacency'
]
for ax, metric in zip(axes, metric_columns):
    values = [metric_df.loc[metric_df.intervention == family, metric].values for family in families]
    ax.boxplot(values, labels=families, showmeans=True)
    ax.tick_params(axis='x', rotation=25); ax.set_title(metric); ax.set_ylabel('performance')
finish_figure('10_ranking_intervention_metrics')

margin_columns = ['image_masked_mean_margin', 'text_masked_mean_margin', 'final_mean_margin']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, margin_columns):
    values = [margin_df.loc[margin_df.intervention == family, column].values for family in families]
    ax.boxplot(values, labels=families, showmeans=True)
    ax.tick_params(axis='x', rotation=25); ax.set_title(column); ax.axhline(0, color='black', linewidth=1)
finish_figure('11_masked_and_final_margin_interventions')

In [ ]:
original_result = next(r for r in intervention_results if r['intervention'] == 'original')
ranking_change_rows = []
for result in intervention_results:
    if result is original_result: continue
    changes = compare_rankings(original_result['rankings'], result['rankings'], config['topk'])
    for top_label, values in changes.items():
        ranking_change_rows.append({
            'intervention': result['intervention'], 'seed': result['seed'],
            'topk': top_label, **values,
        })
ranking_change_df = pd.DataFrame(ranking_change_rows)
display(ranking_change_df)

plot_change = ranking_change_df.groupby('intervention')[
    ['changed_order_rate', 'changed_item_set_rate', 'mean_item_overlap_rate']
].mean().reindex(
    (['constant_mask'] if model.mask_graph_mode == 'soft' else [])
    + ['permute_image', 'permute_text', 'swapped', 'full_adjacency']
)
plot_change.plot(kind='bar', figsize=(11, 4))
plt.ylim(0, 1); plt.ylabel('rate'); plt.title('Top-K list changes relative to original')
plt.xticks(rotation=20)
finish_figure('12_topk_list_changes')

## 6. Gate có thực sự sử dụng masked branch không?
Gate gần 1 ưu tiên full branch; `1-gate` mới là masked-branch weight. Ta xem phân bố gate và liên hệ với keep rate, activity, cũng như mức giảm final triplet margin khi permutation. Đây không phải image–text preference gate.

In [ ]:
gates = collect_gate_snapshot(model)
gate_table = pd.DataFrame({name: probability_summary(values) for name, values in gates.items()}).T
display(gate_table)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for modality, color in [('image', 'tab:blue'), ('text', 'tab:orange')]:
    axes[0].hist(gates[f'{modality}_user'], bins=40, alpha=.55, label=modality, color=color)
    axes[1].hist(gates[f'{modality}_item'], bins=40, alpha=.55, label=modality, color=color)
axes[0].set(title='User full–masked gates', xlabel='mean gate (1 = full)', ylabel='entities')
axes[1].set(title='Item full–masked gates', xlabel='mean gate (1 = full)', ylabel='entities')
axes[0].legend(); axes[1].legend()
finish_figure('13_gate_distributions')

triplet_users = triplets[0].detach().cpu().numpy()
original_final_margin = original_result['triplets']['per_triplet']['final'].numpy()
def mean_user_margin_drop(intervention):
    runs = [r for r in intervention_results if r['intervention'] == intervention]
    drops = np.stack([original_final_margin-r['triplets']['per_triplet']['final'].numpy() for r in runs]).mean(axis=0)
    return pd.DataFrame({'user': triplet_users, f'{intervention}_margin_drop': drops}).groupby('user').mean()

gate_user_df = user_df.set_index('user').copy()
gate_user_df['image_gate'] = gates['image_user'][gate_user_df.index]
gate_user_df['text_gate'] = gates['text_user'][gate_user_df.index]
gate_user_df = gate_user_df.join(mean_user_margin_drop('permute_image')).join(
    mean_user_margin_drop('permute_text')
)
display(gate_user_df[[
    'activity', 'image_keep_rate', 'text_keep_rate', 'image_gate', 'text_gate',
    'permute_image_margin_drop', 'permute_text_margin_drop'
]].corr())

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0,0].scatter(gate_user_df.image_keep_rate, 1-gate_user_df.image_gate, s=5, alpha=.2)
axes[0,0].set(xlabel='image keep rate', ylabel='image masked weight (1-g)', title='Image budget vs gate usage')
axes[0,1].scatter(gate_user_df.text_keep_rate, 1-gate_user_df.text_gate, s=5, alpha=.2, color='tab:orange')
axes[0,1].set(xlabel='text keep rate', ylabel='text masked weight (1-g)', title='Text budget vs gate usage')
axes[1,0].scatter(1-gate_user_df.image_gate, gate_user_df.permute_image_margin_drop, s=5, alpha=.2)
axes[1,0].axhline(0, color='black', linewidth=1); axes[1,0].set(xlabel='image masked weight (1-g)', ylabel='original - permuted margin', title='Image gate vs permutation damage')
axes[1,1].scatter(1-gate_user_df.text_gate, gate_user_df.permute_text_margin_drop, s=5, alpha=.2, color='tab:orange')
axes[1,1].axhline(0, color='black', linewidth=1); axes[1,1].set(xlabel='text masked weight (1-g)', ylabel='original - permuted margin', title='Text gate vs permutation damage')
finish_figure('14_gate_budget_and_permutation_relationships')

## 7. Lưu bảng và summary
Diễn giải chính: permutation giảm masked-only nhưng final không giảm cho thấy full/gate/I–I đang bù; swap giảm cho thấy mask có tính đặc thù modality; permutation không giảm thì chưa có bằng chứng lựa chọn cạnh cụ thể hữu ích. Luôn xem độ lệch qua nhiều seed thay vì một lần.

In [ ]:
probability_table.to_csv(output_dir / 'mask_probability_summary.csv')
user_df.to_csv(output_dir / 'per_user_mask_statistics.csv', index=False)
comparison_table.to_csv(output_dir / 'edge_group_vs_random.csv')
random_df.to_csv(output_dir / 'random_mask_repeats.csv', index=False)
edge_df.to_csv(output_dir / 'edge_characteristics.csv', index=False)
metric_df.to_csv(output_dir / 'intervention_metrics.csv', index=False)
margin_df.to_csv(output_dir / 'intervention_triplet_margins.csv', index=False)
ranking_change_df.to_csv(output_dir / 'intervention_ranking_changes.csv', index=False)
gate_table.to_csv(output_dir / 'gate_summary.csv')
gate_user_df.to_csv(output_dir / 'gate_user_relationships.csv')
soft_activity_summary.to_csv(output_dir / 'soft_weight_by_activity.csv')
if model.mask_graph_mode == 'soft':
    learned_vs_constant.to_csv(output_dir / 'learned_vs_constant_mask.csv')

summary = {
    'checkpoint': str(checkpoint_path), 'dataset': config['dataset'], 'split': SPLIT,
    'selection_kind': snapshot['selection_kind'],
    'probability_summary': probability_table.to_dict(orient='index'),
    'global_edge_group_rates': user_stats['global_group_rates'],
    'global_probability_correlation': user_stats['global_probability_correlation'],
    'zero_edge_user_rate': {
        'image': user_stats['image_zero_edge_user_rate'],
        'text': user_stats['text_zero_edge_user_rate'],
        'both': float(user_df.both_empty.mean()),
    },
    'intervention_metrics': metric_rows,
    'intervention_margins': margin_rows,
}
with open(output_dir / 'analysis_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print('Đã lưu toàn bộ kết quả tại:', output_dir)